# Path-count Structural Analysis

This notebook is a lightweight viewer for path-count artifacts generated by scripts.
It should not contain hidden experiment logic.

Required command:

```bash
PYTHONPATH=src .venv/bin/python scripts/structural_analysis.py
PYTHONPATH=src .venv/bin/python scripts/router_analysis.py --skip-router
```

The final full comparison figure includes `AnyBURL`, `ConvE`, `HoGRN`, `PathBSR`, and `TransE` on all six datasets.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS = PROJECT_ROOT / "results"

MODEL_ORDER = ["AnyBURL", "ConvE", "HoGRN", "PathBSR", "TransE"]
DATASET_ORDER = ["FB15K-237-10", "FB15K-237-20", "FB15K-237-50", "NELL23K", "WD-singer", "WN18RR"]

In [ ]:
summary_path = RESULTS / "path_count/all_datasets_pathcount_mrr_by_model.csv"
figure_path = RESULTS / "path_count/all_datasets_pathcount_mrr_by_model.png"
if not summary_path.exists() or not figure_path.exists():
    display(Markdown("""
**Missing generated artifact.** Run:

```bash
PYTHONPATH=src .venv/bin/python scripts/structural_analysis.py
PYTHONPATH=src .venv/bin/python scripts/router_analysis.py --skip-router
```
"""))
else:
    df = pd.read_csv(summary_path)
    df["dataset"] = pd.Categorical(df["dataset"], DATASET_ORDER, ordered=True)
    df["model"] = pd.Categorical(df["model"], MODEL_ORDER, ordered=True)
    display(df.sort_values(["dataset", "path_bucket", "model"]).head())
    display(Image(filename=str(figure_path)))

In [ ]:
# Full path-count MRR table: dataset x bucket x model.
if summary_path.exists():
    pivot = (
        pd.read_csv(summary_path)
        .pivot_table(index=["dataset", "path_bucket"], columns="model", values="mrr", aggfunc="first")
        .reindex(columns=MODEL_ORDER)
    )
    display(pivot.round(4))

In [ ]:
# Query-share table for the FB sparsity splits.
share_path = RESULTS / "path_count/fb_pathbucket_query_share.csv"
if share_path.exists():
    display(pd.read_csv(share_path).round(3))

In [ ]:
# Existing paper-focused path-count figures.
for name in [
    "fb_pathcount_mrr_grouped_bar.png",
    "nell_wd_pathcount_mrr.png",
    "nell_wd_few_path_margin.png",
]:
    path = RESULTS / "path_count" / name
    if path.exists():
        display(Markdown(f"### `{name}`"))
        display(Image(filename=str(path)))